# Repo-on-Postgres CLI Smoke Test

This notebook uses the current repo as a real test case for `vfs`.

It does three things:

1. Creates a local PostgreSQL database.
2. Loads this repo into a `DatabaseFileSystem` mounted at `/repo`.
3. Exercises `VFSClient.cli(...)` for `read`, `write`, `edit`, `grep`, `glob`, `tree`, and a chained query.

All writes and edits go to the database copy only. The notebook never writes back to the repo files on disk.

## Prereqs

- Local PostgreSQL running and reachable at `postgresql+asyncpg://localhost/postgres`
- `asyncpg` installed, for example with `uv sync --extra postgres`
- Run the notebook from the repo root or from `examples/`

In [1]:
from dataclasses import dataclass
from pathlib import Path
import re
import sys

from sqlalchemy import text
from sqlalchemy.engine import URL, make_url
from sqlalchemy.ext.asyncio import create_async_engine
from sqlmodel import SQLModel

repo_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((path.resolve() for path in repo_candidates if (path / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repo root or from examples/")
sys.path.insert(0, str(REPO_ROOT / "src"))

from vfs.backends.database import DatabaseFileSystem
from vfs.client import VFSClient
from vfs.models import VFSObject

SKIP_DIRS = {
    ".git",
    ".venv",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".ipynb_checkpoints",
    "node_modules",
}

SKIP_SUFFIXES = {
    ".pyc",
    ".pyo",
    ".so",
    ".db",
    ".sqlite",
    ".sqlite3",
    ".png",
    ".jpg",
    ".jpeg",
    ".gif",
    ".ico",
    ".woff",
    ".woff2",
    ".ttf",
    ".eot",
    ".ipynb",
    ".DS_Store",
}


@dataclass(frozen=True)
class RepoSourceFile:
    disk_path: Path
    relative_path: str
    virtual_path: str
    size_bytes: int
    content: str


@dataclass(frozen=True)
class RepoLoadSummary:
    db_name: str
    db_url: str
    mount_root: str
    loaded_files: int
    skipped_files: int
    total_bytes: int


@dataclass(frozen=True)
class SmokeStep:
    label: str
    query: str
    rendered: str


@dataclass(frozen=True)
class RepoCase:
    repo_root: Path
    db_name: str = "vfs_repo_case"
    admin_url: str = "postgresql+asyncpg://localhost/postgres"
    mount_name: str = "repo"
    scratch_dir: str = "__db_only__"
    max_file_bytes: int = 512 * 1024

    def __post_init__(self) -> None:
        object.__setattr__(self, "repo_root", self.repo_root.resolve())
        if re.fullmatch(r"[a-z][a-z0-9_]*", self.db_name) is None:
            raise ValueError("db_name must match [a-z][a-z0-9_]* so CREATE/DROP DATABASE stays safe")
        mount = self.mount_name.strip("/")
        if not mount or "/" in mount:
            raise ValueError("mount_name must be a single non-empty path segment")
        object.__setattr__(self, "mount_name", mount)

    @property
    def admin_engine_url(self) -> URL:
        return make_url(self.admin_url)

    @property
    def database_url(self) -> URL:
        return self.admin_engine_url.set(database=self.db_name)

    @property
    def database_url_str(self) -> str:
        return self.database_url.render_as_string(hide_password=False)

    @property
    def mount_root(self) -> str:
        return f"/{self.mount_name}"

    @property
    def scratch_root(self) -> str:
        return f"{self.mount_root}/{self.scratch_dir}"

    def disk_path(self, relative_path: str | Path) -> Path:
        return self.repo_root / Path(relative_path)

    def virtual_path(self, relative_path: str | Path) -> str:
        rel = Path(relative_path).as_posix().lstrip("/")
        return self.mount_root if not rel else f"{self.mount_root}/{rel}"


async def create_database(case: RepoCase) -> None:
    engine = create_async_engine(case.admin_engine_url, echo=False)
    try:
        async with engine.connect() as raw_conn:
            conn = await raw_conn.execution_options(isolation_level="AUTOCOMMIT")
            result = await conn.execute(
                text("SELECT 1 FROM pg_database WHERE datname = :db_name"),
                {"db_name": case.db_name},
            )
            if result.scalar_one_or_none() is None:
                await conn.execute(text(f"CREATE DATABASE {case.db_name}"))
    finally:
        await engine.dispose()


async def create_schema(case: RepoCase) -> None:
    engine = create_async_engine(case.database_url, echo=False)
    try:
        async with engine.begin() as conn:
            await conn.run_sync(SQLModel.metadata.create_all)
    finally:
        await engine.dispose()


async def drop_database(case: RepoCase) -> None:
    engine = create_async_engine(case.admin_engine_url, echo=False)
    try:
        async with engine.connect() as raw_conn:
            conn = await raw_conn.execution_options(isolation_level="AUTOCOMMIT")
            await conn.execute(
                text(
                    """
                    SELECT pg_terminate_backend(pid)
                    FROM pg_stat_activity
                    WHERE datname = :db_name
                      AND pid <> pg_backend_pid()
                    """
                ),
                {"db_name": case.db_name},
            )
            await conn.execute(text(f"DROP DATABASE IF EXISTS {case.db_name}"))
    finally:
        await engine.dispose()


async def recreate_database(case: RepoCase) -> None:
    await drop_database(case)
    await create_database(case)
    await create_schema(case)


def discover_repo_files(case: RepoCase) -> tuple[list[RepoSourceFile], list[str]]:
    files: list[RepoSourceFile] = []
    skipped: list[str] = []

    for disk_path in sorted(case.repo_root.rglob("*")):
        if not disk_path.is_file():
            continue

        rel_path = disk_path.relative_to(case.repo_root)
        rel_text = rel_path.as_posix()

        if any(part in SKIP_DIRS for part in rel_path.parts):
            skipped.append(rel_text)
            continue

        if disk_path.suffix.lower() in SKIP_SUFFIXES:
            skipped.append(rel_text)
            continue

        try:
            size_bytes = disk_path.stat().st_size
        except OSError:
            skipped.append(rel_text)
            continue

        if size_bytes > case.max_file_bytes:
            skipped.append(rel_text)
            continue

        try:
            content = disk_path.read_text(encoding="utf-8")
        except (OSError, UnicodeDecodeError):
            skipped.append(rel_text)
            continue

        files.append(
            RepoSourceFile(
                disk_path=disk_path,
                relative_path=rel_text,
                virtual_path=case.virtual_path(rel_text),
                size_bytes=size_bytes,
                content=content,
            )
        )

    return files, skipped


def open_client(case: RepoCase) -> VFSClient:
    client = VFSClient()

    async def _build_filesystem() -> DatabaseFileSystem:
        # Build the engine on the client's private loop so asyncpg stays on one loop.
        engine = create_async_engine(case.database_url, echo=False)
        async with engine.begin() as conn:
            await conn.run_sync(SQLModel.metadata.create_all)
        return DatabaseFileSystem(engine=engine)

    dbfs = client._run(_build_filesystem())
    client.add_mount(case.mount_name, dbfs)
    return client


def load_repo_into_db(case: RepoCase, client: VFSClient, *, batch_size: int = 100) -> RepoLoadSummary:
    files, skipped = discover_repo_files(case)
    total_bytes = 0

    for start in range(0, len(files), batch_size):
        batch = files[start : start + batch_size]
        objects = [VFSObject(path=item.virtual_path, content=item.content) for item in batch]
        client.write(objects=objects)
        total_bytes += sum(item.size_bytes for item in batch)

    return RepoLoadSummary(
        db_name=case.db_name,
        db_url=case.database_url_str,
        mount_root=case.mount_root,
        loaded_files=len(files),
        skipped_files=len(skipped),
        total_bytes=total_bytes,
    )


def run_cli_smoke_test(case: RepoCase, client: VFSClient) -> list[SmokeStep]:
    steps: list[SmokeStep] = []

    def capture(label: str, query: str) -> str:
        rendered = client.cli(query)
        steps.append(SmokeStep(label=label, query=query, rendered=rendered))
        return rendered

    readme_disk = case.disk_path("README.md").read_text(encoding="utf-8")
    assert capture("read repo README from the DB copy", f"read {case.virtual_path('README.md')}") == readme_disk

    scratch_virtual = f"{case.scratch_root}/cli_smoke.txt"
    scratch_disk = case.repo_root / case.scratch_dir / "cli_smoke.txt"

    assert not scratch_disk.exists(), "scratch file already exists on disk; expected DB-only path"
    assert capture(
        "write a DB-only scratch file",
        f'write {scratch_virtual} "alpha beta gamma"',
    ) == f"Wrote {scratch_virtual}"
    assert client.cli(f"read {scratch_virtual}") == "alpha beta gamma"
    assert not scratch_disk.exists(), "write unexpectedly created a real repo file"

    assert capture(
        "edit the DB-only scratch file",
        f"edit {scratch_virtual} beta BETA",
    ) == f"Edited {scratch_virtual}"
    assert client.cli(f"read {scratch_virtual}") == "alpha BETA gamma"
    assert not scratch_disk.exists(), "edit unexpectedly modified a real repo file"

    glob_output = capture(
        "glob Python files from the repo snapshot",
        f'glob "{case.mount_root}/src/**/*.py" --max-count 5',
    )
    assert f"{case.mount_root}/src/" in glob_output

    grep_output = capture(
        "grep the repo snapshot",
        f'grep "class VFSClient" {case.mount_root}/src --max-count 5',
    )
    assert "class VFSClient" in grep_output

    tree_output = capture(
        "render a tree of the repo snapshot",
        f"tree {case.mount_root}/src --depth 2",
    )
    assert "src" in tree_output and "vfs" in tree_output

    chain_output = capture(
        "chain glob | grep | read on the DB-only file",
        f'glob "{case.scratch_root}/*.txt" | grep "BETA" | read',
    )
    assert chain_output == "alpha BETA gamma"
    assert case.disk_path("README.md").read_text(encoding="utf-8") == readme_disk

    return steps


In [2]:
CASE = RepoCase(repo_root=REPO_ROOT)
CASE

RepoCase(repo_root=PosixPath('/Users/claygendron/Git/Repos/grover'), db_name='vfs_repo_case', admin_url='postgresql+asyncpg://localhost/postgres', mount_name='repo', scratch_dir='__db_only__', max_file_bytes=524288)

In [3]:
if "client" in globals():
    client.close()

await recreate_database(CASE)
client = open_client(CASE)
summary = load_repo_into_db(CASE, client)
summary

RepoLoadSummary(db_name='vfs_repo_case', db_url='postgresql+asyncpg://localhost/vfs_repo_case', mount_root='/repo', loaded_files=925, skipped_files=90272, total_bytes=7847674)

In [4]:
smoke_steps = run_cli_smoke_test(CASE, client)

for step in smoke_steps:
    print(f"=== {step.label} ===")
    print(step.query)
    print(step.rendered)
    print()

print(f"Completed {len(smoke_steps)} CLI smoke checks against {CASE.mount_root}.")

=== read repo README from the DB copy ===
read /repo/README.md
# vfs: The Virtual File System for Agents

<p align="center">
  <a href="https://pypi.org/project/vfs-py/"><img src="https://img.shields.io/pypi/v/vfs-py" alt="PyPI version"></a>
  <a href="https://pypi.org/project/vfs-py/"><img src="https://img.shields.io/pypi/pyversions/vfs-py" alt="Python"></a>
  <a href="https://github.com/ClayGendron/grover/actions/workflows/test.yml"><img src="https://github.com/ClayGendron/grover/actions/workflows/test.yml/badge.svg" alt="Tests"></a>
  <a href="https://github.com/ClayGendron/grover/blob/main/LICENSE"><img src="https://img.shields.io/github/license/ClayGendron/grover" alt="License"></a>
  <a href="https://codecov.io/gh/ClayGendron/grover"><img src="https://codecov.io/gh/ClayGendron/grover/branch/main/graph/badge.svg" alt="Coverage"></a>
</p>

```bash
pip install vfs-py
```

`vfs` is an in-process file system that mounts data from multiple sources to enable agentic search and operation

## Optional: Try Your Own CLI Queries

Examples:

- `client.cli('read /repo/pyproject.toml')`
- `client.cli('glob "/repo/tests/**/*.py" --max-count 20')`
- `client.cli('grep "DatabaseFileSystem" /repo/src --max-count 10')`
- `client.cli('glob "/repo/src/**/*.py" | grep "create_async_engine" | read')`

In [6]:
print(client.cli('grep "DatabaseFileSystem" /repo/src --max-count 10'))

/repo/src/vfs/backends/__init__.py:3:from vfs.backends.database import DatabaseFileSystem
/repo/src/vfs/backends/__init__.py:6:__all__ = ["DatabaseFileSystem", "MSSQLFileSystem"]
/repo/src/vfs/backends/database.py:1:"""DatabaseFileSystem — SQL-backed implementation of VirtualFileSystem.
/repo/src/vfs/backends/database.py:140:class DatabaseFileSystem(VirtualFileSystem):
/repo/src/vfs/backends/mssql.py:3:Subclass of :class:`DatabaseFileSystem` that overrides the three search
/repo/src/vfs/backends/mssql.py:8:Why this exists: the base ``DatabaseFileSystem`` implementations either
/repo/src/vfs/backends/mssql.py:34:    DatabaseFileSystem,
/repo/src/vfs/backends/mssql.py:105:class MSSQLFileSystem(DatabaseFileSystem):
/repo/src/vfs/backends/mssql.py:109:    search unchanged from :class:`DatabaseFileSystem`.  Only the three
/repo/src/vfs/base.py:11:- ``DatabaseFileSystem`` — SQL via ``VFSObject``
/repo/src/vfs/client.py:13:    await g.add_mount("data", DatabaseFileSystem(engine=engine))
/repo

In [ ]:
DROP_DATABASE = False

if DROP_DATABASE:
    if "client" in globals():
        client.close()
    await drop_database(CASE)
    print(f"Dropped {CASE.db_name}.")
else:
    print(
        f"{CASE.db_name!r} is still available at {CASE.database_url_str}.\n"
        "Set DROP_DATABASE = True and rerun this cell when you want to tear it all down."
    )
